Tools

a name,description,helps llm and ....

In [4]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv()
model = init_chat_model("groq:openai/gpt-oss-120b")
response=model.invoke("hi chat")
response

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'reasoning_content': 'We need to respond as ChatGPT. The user says "hi chat". We should greet and ask how we can help. No policy issues.'}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 73, 'total_tokens': 121, 'completion_time': 0.100645405, 'completion_tokens_details': {'reasoning_tokens': 30}, 'prompt_time': 0.025604718, 'prompt_tokens_details': None, 'queue_time': 0.375688539, 'total_time': 0.126250123}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_9241e9962b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a034e2-84aa-7882-b8ff-759c457f722a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 73, 'output_tokens': 48, 'total_tokens': 121, 'output_token_details': {'reasoning': 30}})

In [5]:
from langchain.tools import tool

@tool
def getWeather(location):
    """Get the weather at a location"""
    return f"It feels like Cool weather in {location}" 
model_with_tools = model.bind_tools([getWeather])

In [7]:
response=model_with_tools.invoke("what's the weather like in New york ?")
print(response)
for tool_call in response.tool_calls:
    # view tool calls made bt the model
    print(f"Tool :{tool_call['name']}")
    print(f"Args: {tool_call['args']}")
    print(tool_call)

content='' additional_kwargs={'reasoning_content': 'User asks weather in New York. Need to call function getWeather with location "New York".', 'tool_calls': [{'id': 'fc_c60cef79-e51c-494d-84d6-24327bb6f2e3', 'function': {'arguments': '{"location":"New York"}', 'name': 'getWeather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 129, 'total_tokens': 177, 'completion_time': 0.101108533, 'completion_tokens_details': {'reasoning_tokens': 20}, 'prompt_time': 0.00552909, 'prompt_tokens_details': None, 'queue_time': 0.375126549, 'total_time': 0.106637623}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_017482bd7f', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a034f7-b0f2-7872-8ae2-bc1c626094a9-0' tool_calls=[{'name': 'getWeather', 'args': {'location': 'New York'}, 'id': 'fc_c60cef79-e51c-494d-84d6-24327bb6f2e3', 'type': 'tool_call'}] invalid_tool_calls

Tools Execution Loops

In [11]:
# step 1: Model Generates tool calls
messages =[{"role":"user","content":"what was the weather in New York ?"}]
ai_msg =model_with_tools.invoke(messages)
messages.append(ai_msg)

#step 2 : Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = getWeather.invoke(tool_call)
    messages.append(tool_result)

#step 3 : Pass results back to model for final Response
final_response=model_with_tools.invoke(messages)
print(final_response.text)
print(messages) 

The latest weather report for New York indicates **cool conditions**. Let me know if you’d like a more detailed forecast (e.g., temperature, precipitation, or a multi‑day outlook).
[{'role': 'user', 'content': 'what was the weather in New York ?'}, AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks "what was the weather in New York?" No specific date/time. Likely they want current weather. We need to fetch current weather via getWeather function. Provide location as "New York". Use function.', 'tool_calls': [{'id': 'fc_3a167e2c-365a-42ae-a5d7-8b83f418107d', 'function': {'arguments': '{"location":"New York"}', 'name': 'getWeather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 128, 'total_tokens': 201, 'completion_time': 0.157224843, 'completion_tokens_details': {'reasoning_tokens': 45}, 'prompt_time': 0.005889082, 'prompt_tokens_details': None, 'queue_time': 0.314853705, 'total_time': 0.163113925}, 'model_name